# Regular Expressions, with Pandas & file loading

### A hands-on tutorial

Regular expressions ("regex") are a mini-language for **describing patterns in text** —
"a 4-digit year", "an email address", "a price". You use them to *find*, *extract*,
*validate*, and *replace* parts of strings.

This notebook is **general-purpose** (no NLP topics). It builds up in three stages:

1. **Regex basics in pure Python** — the `re` module, piece by piece
2. **The building blocks** — character classes, quantifiers, groups, anchors
3. **Loading a real text file** and cleaning it with regex + **Pandas**

> Run every cell (Shift+Enter). A messy sample file `contacts.txt` was shipped next to
> this notebook; the file-loading section reads it. Solutions to exercises are at the end.

In [ ]:
import re
import pandas as pd

print("re module ready")
print("pandas", pd.__version__)

re module ready
pandas 3.0.2


## 1. Regex basics — the five core functions

The `re` module gives you five functions you will use constantly:

| Function | What it does |
|----------|--------------|
| `re.search`  | find the **first** match anywhere (or `None`) |
| `re.match`   | match only at the **start** of the string |
| `re.findall` | return **all** matches as a list |
| `re.sub`     | **replace** matches with something else |
| `re.split`   | **split** a string on a pattern |

Always write patterns as **raw strings**: `r"\d+"`, not `"\d+"`. The `r` stops Python
from eating the backslashes before the regex engine sees them.

In [ ]:
text = "Order 42 shipped on 2023-05-17 for $1,200."

# search: first match anywhere, returns a Match object (or None)
m = re.search(r"\d+", text)
print("search  :", m)
print("  .group():", m.group(), "| .start():", m.start(), "| .end():", m.end())

# match: only at the very start -> None here (text starts with 'Order')
print("match   :", re.match(r"\d+", text))

# findall: every match as a list of strings
print("findall :", re.findall(r"\d+", text))

search  : <re.Match object; span=(6, 8), match='42'>
  .group(): 42 | .start(): 6 | .end(): 8
match   : None
findall : ['42', '2023', '05', '17', '1', '200']


In [ ]:
# sub: replace every match
print(re.sub(r"\d+", "#", text))        # mask every run of digits

# split: break the string wherever the pattern occurs
print(re.split(r"\s+", "one   two\tthree  four"))   # split on any whitespace

Order # shipped on #-#-# for $#,#.
['one', 'two', 'three', 'four']


> **Your turn (1).** From the string `"cat 3 dog 7 fox 11"`, use `re.findall` to pull
> out every number. (Solution at the end.)

## 2. The building blocks

### 2.1 Character classes — *which* characters

| Pattern | Matches |
|---------|---------|
| `.`       | any character except newline |
| `\d` `\D` | a digit / a non-digit |
| `\w` `\W` | a word char `[A-Za-z0-9_]` / a non-word char |
| `\s` `\S` | whitespace / non-whitespace |
| `[aeiou]` | any one listed character |
| `[^aeiou]`| any one character **not** listed |
| `[a-z]` `[0-9]` | a range |

In [ ]:
s = "Room B7, floor 3!"
print("digits     :", re.findall(r"\d", s))
print("word chars :", re.findall(r"\w", s))
print("vowels     :", re.findall(r"[aeiou]", s))
print("NOT vowels :", re.findall(r"[^aeiou ]", s))   # also excludes the space
print("letters A-Z:", re.findall(r"[A-Z]", s))

digits     : ['7', '3']
word chars : ['R', 'o', 'o', 'm', 'B', '7', 'f', 'l', 'o', 'o', 'r', '3']
vowels     : ['o', 'o', 'o', 'o']
NOT vowels : ['R', 'm', 'B', '7', ',', 'f', 'l', 'r', '3', '!']
letters A-Z: ['R', 'B']


### 2.2 Quantifiers — *how many*

| Pattern | Meaning |
|---------|---------|
| `*`     | 0 or more |
| `+`     | 1 or more |
| `?`     | 0 or 1 (optional) |
| `{n}`   | exactly n |
| `{n,m}` | between n and m |
| `{n,}`  | n or more |

In [ ]:
print("one+ digits :", re.findall(r"\d+", "a1 bb22 ccc333"))
print("exactly 4    :", re.findall(r"\d{4}", "12 345 6789 12345"))   # 4-digit runs
print("2 to 3       :", re.findall(r"\d{2,3}", "1 22 333 4444"))
print("optional 's' :", re.findall(r"cats?", "cat cats cat"))        # cat or cats

one+ digits : ['1', '22', '333']
exactly 4    : ['6789', '1234']
2 to 3       : ['22', '333', '444']
optional 's' : ['cat', 'cats', 'cat']


### 2.3 Greedy vs lazy — a classic trap

By default quantifiers are **greedy** — they grab as much as possible. Add `?` to make
them **lazy** — grab as little as possible. This matters constantly.

In [ ]:
html = "<a><b><c>"
print("greedy <.*> :", re.findall(r"<.*>", html))    # one giant match
print("lazy  <.*?>:", re.findall(r"<.*?>", html))    # three small matches

greedy <.*> : ['<a><b><c>']
lazy  <.*?>: ['<a>', '<b>', '<c>']


### 2.4 Anchors and boundaries — *where*

| Pattern | Meaning |
|---------|---------|
| `^`  | start of string |
| `$`  | end of string |
| `\b` | word boundary (edge of a word) |

In [ ]:
print("starts with Hi? :", bool(re.match(r"^Hi", "Hi there")))
print("ends with .py?  :", bool(re.search(r"\.py$", "script.py")))

# \b stops 'cat' matching inside 'concatenate'
print("whole-word cat  :", re.findall(r"\bcat\b", "a cat in concatenate"))

starts with Hi? : True
ends with .py?  : True
whole-word cat  : ['cat']


### 2.5 Groups and alternation — *capture* and *or*

- `( ... )` is a **capturing group** — it remembers the matched part.
- `|` means **or**.
- `(?: ... )` groups **without** capturing (when you just need the grouping).

In [ ]:
# alternation: match either word
print("cat or dog :", re.findall(r"cat|dog", "a cat and a dog"))

# capturing groups: findall returns the GROUPS, not the whole match
date = "2023-05-17"
m = re.search(r"(\d{4})-(\d{2})-(\d{2})", date)
print("whole match:", m.group(0))
print("year/mon/day:", m.group(1), m.group(2), m.group(3))

# findall with two groups returns a list of tuples
pairs = re.findall(r"(\w+)=(\d+)", "a=1 b=22 c=333")
print("key=value pairs:", pairs)

cat or dog : ['cat', 'dog']
whole match: 2023-05-17
year/mon/day: 2023 05 17
key=value pairs: [('a', '1'), ('b', '22'), ('c', '333')]


> **Your turn (2).** Write one pattern that matches a time like `09:30` or `14:05`
> (two digits, a colon, two digits) and use `findall` on `"wake 06:30 lunch 13:00"`.
> (Solution at the end.)

## 3. A few practical patterns

These are the everyday ones. They are *good enough* for cleaning data — not strict
specifications, and that is fine. (A perfect email regex, for instance, is monstrous;
you rarely need one.)

In [ ]:
samples = [
    "reach me at ada@math.org or alan.turing@cs.ac.uk",
    "call +92-300-1234567 or 0301 7654321",
    "paid $1,200.50 and $840 on 2023-05-17",
]

email = r"[\w.]+@[\w.]+"
phone = r"\+?\d[\d\s()-]{7,}\d"
price = r"\$[\d,]+(?:\.\d{2})?"
date  = r"\d{4}-\d{2}-\d{2}"

print("emails :", re.findall(email, samples[0]))
print("phones :", re.findall(phone, samples[1]))
print("prices :", re.findall(price, samples[2]))
print("dates  :", re.findall(date,  samples[2]))

emails : ['ada@math.org', 'alan.turing@cs.ac.uk']
phones : ['+92-300-1234567', '0301 7654321']
prices : ['$1,200.50', '$840']
dates  : ['2023-05-17']


### Compiling a pattern

If you use a pattern many times (e.g. once per row of a big file), **compile it once**
with `re.compile` for speed and readability.

In [ ]:
price_re = re.compile(r"\$[\d,]+(?:\.\d{2})?")

for line in ["spent $55.25 today", "nothing here", "two: $9,999 and $3.00"]:
    print(line, "->", price_re.findall(line))

spent $55.25 today -> ['$55.25']
nothing here -> []
two: $9,999 and $3.00 -> ['$9,999', '$3.00']


## 4. Loading a text file

Now the practical part: read a real file from disk and work on it. A messy file,
`contacts.txt`, was shipped next to this notebook. It has two `#` comment lines at the
top, a header, then one record per line with fields separated by `|`. First, let us look
at it the plain Python way, **before** bringing in Pandas.

In [ ]:
# read the whole file and print it as-is
with open("contacts.txt") as f:
    raw = f.read()

print(raw)

# customer contact log  (messy on purpose)
# one record per line; fields are separated by the pipe character |
id|raw_record
1|Ada Lovelace , ada@math.org , +92-300-1234567 , joined 2021-03-15 , spent $1,200.50
2|Alan Turing , alan.turing@cs.ac.uk , 0301 7654321 , joined 2019/11/02 , spent $840
3|Grace Hopper , grace_hopper@navy.mil , +1 (555) 019-2834 , joined 2020-07-21 , spent $2,015.00
4|Linus T. , linus@kernel.org , 03009998877 , joined 2022-01-30 , spent $55.25
5|Katherine J , k.johnson@nasa.gov , +92 321 4455667 , joined 2018-05-09 , spent $9,999
6|BAD ROW no email here , 12345 , joined yesterday , spent lots
7|Margaret H , margaret@apollo.org , +92-333-0001112 , joined 2023-12-01 , spent $3.00



In [ ]:
# read line by line into a list, dropping comment lines and blanks
with open("contacts.txt") as f:
    lines = [ln.strip() for ln in f
             if ln.strip() and not ln.startswith("#")]

print("number of lines kept (incl. header):", len(lines))
print("header      :", lines[0])
print("first record:", lines[1])

number of lines kept (incl. header): 8
header      : id|raw_record
first record: 1|Ada Lovelace , ada@math.org , +92-300-1234567 , joined 2021-03-15 , spent $1,200.50


## 5. Loading into Pandas and cleaning with regex

Pandas can read the file directly. We tell `read_csv` three things: the separator is `|`
(`sep="|"`), lines starting with `#` are comments (`comment="#"`), and read every column
as text (`dtype=str`) so the regex `.str` methods work on them. Each row's useful content
is crammed into one messy `raw_record` column — exactly the situation regex is built for.

In [ ]:
df = pd.read_csv("contacts.txt", sep="|", comment="#", dtype=str)
df

  id                                                                                      raw_record
0  1             Ada Lovelace , ada@math.org , +92-300-1234567 , joined 2021-03-15 , spent $1,200.50
1  2              Alan Turing , alan.turing@cs.ac.uk , 0301 7654321 , joined 2019/11/02 , spent $840
2  3  Grace Hopper , grace_hopper@navy.mil , +1 (555) 019-2834 , joined 2020-07-21 , spent $2,015.00
3  4                    Linus T. , linus@kernel.org , 03009998877 , joined 2022-01-30 , spent $55.25
4  5           Katherine J , k.johnson@nasa.gov , +92 321 4455667 , joined 2018-05-09 , spent $9,999
5  6                                   BAD ROW no email here , 12345 , joined yesterday , spent lots
6  7            Margaret H , margaret@apollo.org , +92-333-0001112 , joined 2023-12-01 , spent $3.00

### Pandas string methods take regex directly

Pandas exposes vectorised string operations under `.str`, and most of them accept a
regex. No Python loop needed — the pattern is applied to **every row at once**.

`Series.str.extract(pattern)` pulls out the **first** capturing group per row into a new
column. This is the workhorse for turning one messy column into many clean ones.

In [ ]:
col = df["raw_record"]

# pull each field into its own column using one regex per field
df["email"] = col.str.extract(r"([\w.]+@[\w.]+)")
df["price"] = col.str.extract(r"\$([\d,]+(?:\.\d{2})?)")
df["date"]  = col.str.extract(r"(\d{4}[-/]\d{2}[-/]\d{2})")

df[["id", "email", "price", "date"]]

  id                  email     price        date
0  1           ada@math.org  1,200.50  2021-03-15
1  2   alan.turing@cs.ac.uk       840  2019/11/02
2  3  grace_hopper@navy.mil  2,015.00  2020-07-21
3  4       linus@kernel.org     55.25  2022-01-30
4  5     k.johnson@nasa.gov     9,999  2018-05-09
5  6                    NaN       NaN         NaN
6  7    margaret@apollo.org      3.00  2023-12-01

Notice **row 6** (the deliberately broken record): it has no email, no price, no
valid date, so `extract` returns `NaN` (missing) for those cells instead of crashing.
That graceful handling of non-matches is exactly why `extract` is so useful on real,
dirty data.

### `str.contains` — filter rows by a pattern

`str.contains(pattern)` returns a boolean per row, which you use as a mask — just like
filtering in the Pandas you already know.

In [ ]:
# keep only rows that contain a valid-looking email
has_email = df["raw_record"].str.contains(r"[\w.]+@[\w.]+", regex=True)
print("rows with an email:", int(has_email.sum()), "of", len(df))

print("\ngovernment/academic addresses (.gov/.ac/.mil/.edu):")
gov = df["email"].str.contains(r"\.(?:gov|ac|mil|edu)", regex=True, na=False)
print(df.loc[gov, ["id", "email"]].to_string(index=False))

rows with an email: 6 of 7

government/academic addresses (.gov/.ac/.mil/.edu):
id                 email
 2  alan.turing@cs.ac.uk
 3 grace_hopper@navy.mil
 5    k.johnson@nasa.gov


### `str.replace` with regex — clean values in place

Here we strip the thousands-commas out of the price strings so they can become real
numbers, then convert the column to numeric and do arithmetic.

In [ ]:
# remove commas, convert to float
df["price_num"] = (df["price"]
                   .str.replace(r",", "", regex=True)
                   .astype(float))

print(df[["id", "price", "price_num"]].to_string(index=False))
print("\ntotal spent by matched rows:", df["price_num"].sum())
print("average (ignoring the broken row):", round(df["price_num"].mean(), 2))

id    price  price_num
 1 1,200.50    1200.50
 2      840     840.00
 3 2,015.00    2015.00
 4    55.25      55.25
 5    9,999    9999.00
 6      NaN        NaN
 7     3.00       3.00

total spent by matched rows: 14112.75
average (ignoring the broken row): 2352.12


### `str.extractall` — every match, not just the first

Some rows may contain several matches. `extractall` returns **all** of them, with a
multi-index telling you which row each came from. Here we pull every run of digits from
each record.

In [ ]:
# every group of digits in every record (first 8 shown)
all_digits = df["raw_record"].str.extractall(r"(\d+)")
print(all_digits.head(8).to_string())

               0
  match         
0 0           92
  1          300
  2      1234567
  3         2021
  4           03
  5           15
  6            1
  7          200


### Putting it together — a clean table from a messy file

The whole point: we started with one ugly `raw_record` column and, using only regex +
Pandas string methods, produced a tidy table ready for analysis.

In [ ]:
clean = df[["id", "email", "date", "price_num"]].copy()
clean = clean.rename(columns={"price_num": "spent"})
clean["valid"] = clean["email"].notna() & clean["spent"].notna()
clean

  id                  email        date    spent  valid
0  1           ada@math.org  2021-03-15  1200.50   True
1  2   alan.turing@cs.ac.uk  2019/11/02   840.00   True
2  3  grace_hopper@navy.mil  2020-07-21  2015.00   True
3  4       linus@kernel.org  2022-01-30    55.25   True
4  5     k.johnson@nasa.gov  2018-05-09  9999.00   True
5  6                    NaN         NaN      NaN  False
6  7    margaret@apollo.org  2023-12-01     3.00   True

> **Your turn (3).** Add a new column `domain` to `clean` that contains just the part
> of the email **after** the `@` (e.g. `math.org`). Hint: `str.extract(r"@(...)")`.
> (Solution at the end.)

## 6. Saving the cleaned result

Finally, write the tidy table back out — as CSV (for spreadsheets) and show how you
would read it back. This closes the loop: messy text in, clean data out.

In [ ]:
clean.to_csv("contacts_clean.csv", index=False)
print("wrote contacts_clean.csv")

# read it back to confirm
check = pd.read_csv("contacts_clean.csv")
print(check.to_string(index=False))

wrote contacts_clean.csv
 id                 email       date   spent  valid
  1          ada@math.org 2021-03-15 1200.50   True
  2  alan.turing@cs.ac.uk 2019/11/02  840.00   True
  3 grace_hopper@navy.mil 2020-07-21 2015.00   True
  4      linus@kernel.org 2022-01-30   55.25   True
  5    k.johnson@nasa.gov 2018-05-09 9999.00   True
  6                   NaN        NaN     NaN  False
  7   margaret@apollo.org 2023-12-01    3.00   True


## 7. Exercises

Use only what is in this notebook. Solutions are in the final cell.

1. **Find years.** From `"born 1990, moved 2015, now 2024"`, extract all 4-digit years.
2. **Validate.** Write a function `looks_like_email(s)` returning `True`/`False` using a
   regex and `re.search`. Test it on `"a@b.com"` and `"not-an-email"`.
3. **Mask phone numbers.** In the string `"call 0300-1234567 or 0321-7654321"`, replace
   every phone number with `"[redacted]"` using `re.sub`.
4. **Pandas extract.** Given a Series `pd.Series(["item-42","item-7","nope"])`, extract
   the number after `item-` into a new Series (non-matches become `NaN`).
5. **Challenge.** From `contacts.txt` loaded into Pandas, count how many records have a
   phone number that starts with `+92` (Pakistan). Hint: `str.contains(r"\+92")`.

In [ ]:
# ---------------------------------------------------------------
# SOLUTIONS  (try the exercises yourself first)
# ---------------------------------------------------------------

# --- Your turn (1): numbers ---
print("Your turn 1:", re.findall(r"\d+", "cat 3 dog 7 fox 11"))

# --- Your turn (2): times ---
print("Your turn 2:", re.findall(r"\d{2}:\d{2}", "wake 06:30 lunch 13:00"))

# --- Your turn (3): email domain column ---
clean["domain"] = clean["email"].str.extract(r"@(.+)")
print("\nYour turn 3:")
print(clean[["id", "email", "domain"]].to_string(index=False))

# --- Exercise 1: years ---
print("\nEx 1:", re.findall(r"\d{4}", "born 1990, moved 2015, now 2024"))

# --- Exercise 2: email validator ---
def looks_like_email(s):
    return bool(re.search(r"^[\w.]+@[\w.]+\.\w+$", s))
print("Ex 2:", looks_like_email("a@b.com"), looks_like_email("not-an-email"))

# --- Exercise 3: mask phones ---
masked = re.sub(r"\d{4}-\d{7}", "[redacted]", "call 0300-1234567 or 0321-7654321")
print("Ex 3:", masked)

# --- Exercise 4: pandas extract number ---
s = pd.Series(["item-42", "item-7", "nope"])
print("Ex 4:")
print(s.str.extract(r"item-(\d+)").to_string())

# --- Exercise 5: count +92 phones ---
df2 = pd.read_csv("contacts.txt", sep="|", comment="#", dtype=str)
pk = df2["raw_record"].str.contains(r"\+92", regex=True)
print("\nEx 5: records with +92 phone:", int(pk.sum()))

Your turn 1: ['3', '7', '11']
Your turn 2: ['06:30', '13:00']

Your turn 3:
id                 email     domain
 1          ada@math.org   math.org
 2  alan.turing@cs.ac.uk   cs.ac.uk
 3 grace_hopper@navy.mil   navy.mil
 4      linus@kernel.org kernel.org
 5    k.johnson@nasa.gov   nasa.gov
 6                   NaN        NaN
 7   margaret@apollo.org apollo.org

Ex 1: ['1990', '2015', '2024']
Ex 2: True False
Ex 3: call [redacted] or [redacted]
Ex 4:
     0
0   42
1    7
2  NaN

Ex 5: records with +92 phone: 3


---

### Quick reference card

```
.        any char            \d  digit      \w  word char   \s  whitespace
*        0+                   +   1+         ?   0 or 1
{n}      exactly n            {n,m} n to m    ^   start       $   end
[abc]    one of a,b,c         [^abc] not those           \b  word boundary
(...)    capturing group      (?:...) non-capturing       a|b  a or b
greedy:  .*   (as much as possible)        lazy: .*?  (as little as possible)
```

**Pandas string methods that take regex** (all vectorised, no loop):

```
s.str.contains(pat)    -> bool per row     (filter)
s.str.extract(pat)     -> first group      (one messy col -> clean cols)
s.str.extractall(pat)  -> every match      (multi-index)
s.str.replace(pat, r)  -> clean in place
s.str.findall(pat)     -> list per row
```

### Workflow you just learned
messy text file  →  `pd.read_csv`  →  `str.extract` / `str.replace` with regex
→  clean typed columns  →  `to_csv`.  That is a complete data-cleaning pipeline.

*Tip:* test a regex on one string with `re.findall` first, then apply it to a whole
column with `.str`. Debugging one pattern is far easier than debugging a loop.